<a href="https://colab.research.google.com/github/yoshiagent/IntControl-colab/blob/main/notebooks/NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 類神經網路：原課程 MATLAB 指令的 Python 版

原課程用 MATLAB 的 Neural Network Toolbox（`newp`、`sim`、`init`、`train`、`newff`、`gensim`…）示範感知器與倒傳遞網路。
這本筆記本用 numpy 寫出同樣的功能，逐一重現網頁上的指令與結果。對照的網頁：
[建立感知器](https://yoshiagent.github.io/IntControl/NN-CreatingP.html)、[模擬](https://yoshiagent.github.io/IntControl/NN-SimP.html)、
[初始化](https://yoshiagent.github.io/IntControl/NN-InitP.html)、[訓練](https://yoshiagent.github.io/IntControl/NN-Train.html)、
[NNtool：AND 閘](https://yoshiagent.github.io/IntControl/NN-NNetAnd.html)、[NN2Simulink](https://yoshiagent.github.io/IntControl/NN-NnetGenSim.html)、
[倒傳遞網路](https://yoshiagent.github.io/IntControl/NN-NetBp.html)。

| MATLAB 工具箱 | 這本筆記本 |
|---|---|
| `net = newp(PR, S)` | `net = newp(PR, S)`（自己寫的小類別） |
| `net.IW{1,1}`、`net.b{1}` | `net.IW`、`net.b` |
| `a = sim(net, p)` | `a = sim(net, p)`（p 的每一行是一個輸入向量，和 MATLAB 相同） |
| `net = init(net)` | `init(net)` |
| `net = train(net, p, t)` | `train(net, p, t)`（感知器用 trainc：逐筆循環訓練） |
| `load NnetAnd.mat` | `scipy.io.loadmat`（直接讀原講義的 .mat 檔） |

只用 Colab 內建的 numpy、scipy 與 plotly。

In [1]:
import io
import urllib.request
import numpy as np
import scipy.io as sio
import plotly.graph_objects as go

rng = np.random.default_rng(0)

## 1. 建立感知器：newp

```matlab
net = newp(PR, S)
net = newp([0 2], 1);
inputweights = net.inputweights{1,1}
biases = net.biases{1}
```
`PR` 是 R × 2 的矩陣，每一列是一個輸入的 [最小值 最大值]；`S` 是神經元個數。轉移函數是 hardlim：n ≥ 0 時輸出 1，否則 0。

In [2]:
def hardlim(n):
    return (np.asarray(n) >= 0).astype(int)

class Perceptron:
    """MATLAB newp 建立的網路物件的簡化版：一層、S 個 hardlim 神經元"""
    def __init__(self, PR, S):
        self.PR = np.atleast_2d(np.array(PR, float))
        R = self.PR.shape[0]
        self.IW = np.zeros((S, R))                 # net.IW{1,1}
        self.b = np.zeros(S)                       # net.b{1}
        self.inputweights = dict(delays=0, initFcn="initzero", learn=1, learnFcn="learnp", learnParam=[], size=[S, R], weightFcn="dotprod")
        self.biases = dict(initFcn="initzero", learn=1, learnFcn="learnp", learnParam=[], size=S)
        self.layer = dict(netInputFcn="netsum", transferFcn="hardlim")
        self.trainParam = dict(epochs=100, goal=0)
        self.performFcn = "mae"

def newp(PR, S):
    return Perceptron(PR, S)

def show(d):
    """像 MATLAB 一樣列出結構的欄位"""
    w = max(len(k) for k in d)
    for k, v in d.items():
        print(f"{k:>{w + 4}}: {v!r}")

net = newp([[0, 2]], 1)
print("inputweights ="); show(net.inputweights)
print("biases ="); show(net.biases)

inputweights =
        delays: 0
       initFcn: 'initzero'
         learn: 1
      learnFcn: 'learnp'
    learnParam: []
          size: [1, 1]
     weightFcn: 'dotprod'
biases =
       initFcn: 'initzero'
         learn: 1
      learnFcn: 'learnp'
    learnParam: []
          size: 1


權值函數 `dotprod` 算出 **Wp**，加上偏權值的是層的淨輸入函數 `netsum`（n = Wp + b），最後經過 hardlim 得到輸出。

## 2. 模擬：sim

```matlab
net = newp([-2 2; -2 +2], 1);
net.IW{1,1} = [-1 1];
net.b{1} = [1];
p1 = [1;1];
a1 = sim(net,p1)        % a1 = 1
p2 = [1;-1]
a2 = sim(net,p2)        % a2 = 0
p3 = {[1;1] [1;-1]};
a3 = sim(net,p3)        % a3 = [1]    [0]
```
輸入可以是一個行向量、一個矩陣（每一行是一個輸入，同時送入），或一個 cell 陣列（依序送入的序列）。
Python 的 list 也常拿來寫向量，為了不混淆，cell 陣列用下面的 `cell(...)` 表示。

In [3]:
class Cell(list):
    """MATLAB 的 cell 陣列 {a b c}"""

def cell(*items):
    return Cell(items)

def sim(net, p):
    if isinstance(p, Cell):                        # cell 陣列：依序模擬，回傳 cell
        return Cell(sim(net, q) for q in p)
    p = np.asarray(p, float)
    P = p.reshape(-1, 1) if p.ndim == 1 else p
    a = hardlim(net.IW @ P + net.b[:, None])       # dotprod → netsum → hardlim
    return a.ravel() if p.ndim == 1 else a

net = newp([[-2, 2], [-2, 2]], 1)
net.IW = np.array([[-1.0, 1.0]]); net.b = np.array([1.0])
print("a1 =", sim(net, [1, 1]))
print("a2 =", sim(net, [1, -1]))
a3 = sim(net, cell([1, 1], [1, -1]))                 # p3 = {[1;1] [1;-1]}
print("a3 =", "    ".join(f"[{int(v[0])}]" for v in a3))

a1 = [1]
a2 = [0]
a3 = [1]    [0]


## 3. 初始化：init

```matlab
net = newp([-2 2; -2 +2], 1);
wts = net.IW{1,1}          % 0 0
bias = net.b{1}            % 0
net.IW{1,1} = [3,4];
net.b{1} = 5;
net = init(net);           % 回到 0 0、0
net.inputweights{1,1}.initFcn = 'rands';
net.biases{1}.initFcn = 'rands';
net = init(net);           % 例如 0.2309 0.5839、-0.1106
```
`initzero` 把權值設成 0；`rands` 設成 −1～1 的均勻亂數（亂數不同，數字和原講義不會一樣）。

In [4]:
def init(net):
    S, R = net.IW.shape
    net.IW = rng.uniform(-1, 1, (S, R)) if net.inputweights["initFcn"] == "rands" else np.zeros((S, R))
    net.b = rng.uniform(-1, 1, S) if net.biases["initFcn"] == "rands" else np.zeros(S)

net = newp([[-2, 2], [-2, 2]], 1)
print("wts =", net.IW, " bias =", net.b)
net.IW = np.array([[3.0, 4.0]]); net.b = np.array([5.0])
init(net)
print("init 之後：wts =", net.IW, " bias =", net.b)
net.inputweights["initFcn"] = "rands"; net.biases["initFcn"] = "rands"
init(net)
print("rands：wts =", np.round(net.IW, 4), " biases =", np.round(net.b, 4))

wts = [[0. 0.]]  bias = [0.]
init 之後：wts = [[0. 0.]]  bias = [0.]
rands：wts = [[ 0.2739 -0.4604]]  biases = [-0.9181]


## 4. 訓練：learnp 與 train（trainc）

感知器學習規則 `learnp`：e = t − a，ΔW = e pᵀ，Δb = e。`train` 對感知器預設用 `trainc`：每個回合（epoch）依序拿每一筆資料，算輸出、誤差、立刻修正權值。
回合結束時，若這一回合的平均絕對誤差（mae）達到目標 `goal`（預設 0），就停止並顯示 `Performance goal met.`。

```matlab
net = newp([-2 2; -2 +2], 1);
p = [2; 2];
t = [0];
net.trainParam.epochs = 1;
net = train(net, p, t);      % w = -2 -2，b = -1

net = newp([-2 2; -2 +2], 1);
net.trainParam.epochs = 1;
p = [[2;2] [1;-2] [-2;2] [-1;1]];
t = [0 1 0 1];
net = train(net, p, t);      % w = -3 -1，b = 0
a = sim(net, p)              % a = [0] [0] [1] [1]
```

In [5]:
def learnp(net, p, e):
    net.IW = net.IW + np.outer(e, p)
    net.b = net.b + e

def train(net, P, T, verbose=True):
    P, T = np.atleast_2d(np.asarray(P, float)), np.atleast_2d(np.asarray(T, float))
    epochs, goal = net.trainParam["epochs"], net.trainParam["goal"]
    if verbose:
        print(f"TRAINC, Epoch 0/{epochs}")
    for ep in range(1, epochs + 1):
        errs = []
        for q in range(P.shape[1]):                 # 逐筆循環
            p, t = P[:, q], T[:, q]
            e = t - sim(net, p)
            learnp(net, p, e)
            errs.append(np.abs(e).mean())
        if np.mean(errs) <= goal:
            if verbose:
                print(f"TRAINC, Epoch {ep}/{epochs}\nTRAINC, Performance goal met.")
            return ep
    if verbose:
        print(f"TRAINC, Epoch {epochs}/{epochs}\nTRAINC, Maximum epoch reached.")
    return epochs

net = newp([[-2, 2], [-2, 2]], 1)
net.trainParam["epochs"] = 1
train(net, [[2], [2]], [[0]], verbose=False)
print("w =", net.IW, " b =", net.b)

P = np.array([[2, 1, -2, -1], [2, -2, 2, 1]])
T = np.array([[0, 1, 0, 1]])
net = newp([[-2, 2], [-2, 2]], 1)
net.trainParam["epochs"] = 1
train(net, P, T, verbose=False)
print("w =", net.IW, " b =", net.b)
print("a =", sim(net, P))

w = [[-2. -2.]]  b = [-1.]
w = [[-3. -1.]]  b = [0.]
a = [[0 0 1 1]]


一個回合之後還沒全部分對（第 2、3 筆錯）。把 epochs 設為 20 重新訓練，到第 3 個回合整個回合都沒有誤差，就停止：

```
TRAINC, Epoch 0/20
TRAINC, Epoch 3/20
TRAINC, Performance goal met.
```

In [6]:
net = newp([[-2, 2], [-2, 2]], 1)
net.trainParam["epochs"] = 20
train(net, P, T)
print("w =", net.IW, " b =", net.b, " a =", sim(net, P))

TRAINC, Epoch 0/20
TRAINC, Epoch 3/20
TRAINC, Performance goal met.
w = [[-2. -3.]]  b = [1.]  a = [[0 1 0 1]]


### 決策邊界

訓練好的感知器以直線 W p + b = 0 把平面分成兩半：

In [7]:
def boundary_plot(net, P, T, title, lim=(-3, 3)):
    xs = np.linspace(*lim, 2)
    w1, w2, b = net.IW[0, 0], net.IW[0, 1], net.b[0]
    fig = go.Figure()
    for cls, sym in ((0, "circle-open"), (1, "circle")):
        k = T[0] == cls
        fig.add_scatter(x=P[0, k], y=P[1, k], mode="markers", marker=dict(symbol=sym, size=14, line=dict(width=2)), name=f"target {cls}")
    fig.add_scatter(x=xs, y=-(w1 * xs + b) / w2, mode="lines", name=f"{w1:g} p1 + {w2:g} p2 + {b:g} = 0")
    fig.update_layout(title=title, xaxis=dict(range=list(lim), title="p1"), yaxis=dict(range=list(lim), title="p2", scaleanchor="x"), width=520, height=480)
    return fig

boundary_plot(net, P, T, "perceptron after trainc").show()

## 5. NNtool：AND 閘

老師在 NNtool（圖形介面）建立並訓練了一個 AND 感知器，結果存在附檔 `NnetAnd.mat`。下面先自己訓練一次，再直接下載原講義的 .mat 檔，用 `scipy.io.loadmat` 讀出網路物件裡的權值。

In [8]:
Pand = np.array([[0, 0, 1, 1], [0, 1, 0, 1]])
Tand = np.array([[0, 0, 0, 1]])
net_and = newp([[0, 1], [0, 1]], 1)
train(net_and, Pand, Tand)
print("自己訓練：W =", net_and.IW, " b =", net_and.b, " 輸出", sim(net_and, Pand))

TRAINC, Epoch 0/100
TRAINC, Epoch 6/100
TRAINC, Performance goal met.
自己訓練：W = [[2. 1.]]  b = [-3.]  輸出 [[0 0 0 1]]


In [9]:
URL = "https://yoshiagent.github.io/IntControl/files/IntControl/NNet/"

def load_mat(name):
    """下載原講義的 .mat 檔並讀入（Colab 需要連網）"""
    with urllib.request.urlopen(URL + name, timeout=30) as r:
        return sio.loadmat(io.BytesIO(r.read()))

m = load_mat("NnetAnd.mat")
NetAnd = m["NetAnd"][0, 0]
print("檔案裡的變數：", [k for k in m if not k.startswith("__")])
print("p =", m["p"][0, 0].tolist(), " t =", m["t"][0, 0].tolist())
print("NetAnd.IW{1,1} =", NetAnd["IW"][0, 0], " NetAnd.b{1} =", NetAnd["b"][0, 0].ravel())
print("trainFcn =", NetAnd["trainFcn"][0], "，performFcn =", NetAnd["performFcn"][0], "，netInputFcn =", NetAnd["layers"][0, 0][0, 0]["netInputFcn"][0])

檔案裡的變數： ['NetAnd', 'p', 't']
p = [[0, 0, 1, 1], [0, 1, 0, 1]]  t = [[0, 0, 0, 1]]
NetAnd.IW{1,1} = [[2 1]]  NetAnd.b{1} = [-3]
trainFcn = trainc ，performFcn = mae ，netInputFcn = netsum


檔案裡的網路 W = [2 1]、b = −3，和上面自己從 0 開始依序訓練的結果相同。(1, 1) 的淨輸入剛好是 0，靠 hardlim(0) = 1 才分對，決策邊界正好通過 (1, 1)：

In [10]:
boundary_plot(net_and, Pand, Tand, "AND perceptron (NnetAnd.mat: W = [2 1], b = -3)", lim=(-0.5, 1.5)).show()

## 6. NN2Simulink：gensim 產生的模型

`gensim(NetAnd)` 把網路產生成 Simulink 模型 `NetAndmdl.mdl`。模型的方塊依序是：常數輸入 [0.95; 0.23] → **dotprod**（權值常數 [2; 1]）→ **netsum**（加上偏權值常數 −3）→ **hardlim** → 輸出。
用 Python 照方塊一個一個算：

In [11]:
def NetAndmdl(p=(0.95, 0.23)):
    z = np.dot([2, 1], p)          # Weight Functions/dotprod
    n = z + (-3)                   # Net Input Functions/netsum（加上 b）
    a = int(n >= 0)                # Transfer Functions/hardlim
    print(f"dotprod = {z:.2f}，netsum n = {n:.2f}，hardlim a = {a}")
    return a

NetAndmdl()

dotprod = 2.13，netsum n = -0.87，hardlim a = 0


0

## 7. 倒傳遞網路：NetBp.mat 與 NetBpmdl.mdl

附檔 `NetBp.mat` 是 2 個輸入、3 個 tansig 隱藏神經元、1 個 purelin 輸出的網路，設定和 MATLAB 手冊的範例相同：

```matlab
net = newff([-1 2; 0 5],[3,1],{'tansig','purelin'},'traingd');
p = [-1 -1 2 2; 0 5 0 5];
t = [-1 -1 1 1];
net = train(net, p, t);
```
直接讀檔取出權值，算出網路在手冊四點的輸出與 mse，以及 Simulink 模型 `NetBpmdl.mdl` 的常數輸入 [0.62; 0.79] 的輸出：

In [12]:
nb = load_mat("NetBp.mat")["NetBp"][0, 0]
W1, b1 = nb["IW"][0, 0], nb["b"][0, 0].ravel()
W2, b2 = nb["LW"][1, 0], nb["b"][1, 0].ravel()
print("transferFcn =", [str(L[0, 0]["transferFcn"][0]) for L in nb["layers"].ravel()], "，trainFcn =", nb["trainFcn"][0],
      "，lr =", nb["trainParam"][0, 0]["lr"][0, 0])
print("IW{1,1} =\n", np.round(W1, 4), "\nb{1} =", np.round(b1, 4), "\nLW{2,1} =", np.round(W2, 4), " b{2} =", np.round(b2, 4))

def tansig(n):
    return 2 / (1 + np.exp(-2 * n)) - 1            # 和 tanh 相同

def netbp(p, W1=W1, b1=b1, W2=W2, b2=b2):
    """p 的每一行是一個輸入；回傳輸出列向量"""
    a1 = tansig(W1 @ p + b1[:, None])              # Layer 1（NetBpmdl 用 Goto／From 方塊把 a{1} 傳到 Layer 2）
    return (W2 @ a1 + b2[:, None]).ravel()         # Layer 2：purelin

Pd = np.array([[-1, -1, 2, 2], [0, 5, 0, 5]], float)
Td = np.array([-1, -1, 1, 1], float)
y = netbp(Pd)
print("四點輸出 =", np.round(y, 2), "，mse =", round(float(np.mean((Td - y) ** 2)), 4))
print("NetBpmdl：輸入 [0.62; 0.79] → y =", round(float(netbp(np.array([[0.62], [0.79]]))[0]), 3))

transferFcn = ['tansig', 'purelin'] ，trainFcn = traingd ，lr = 0.01
IW{1,1} =
 [[ 1.6104 -0.0156]
 [-1.1816  0.5496]
 [ 0.6509  0.8971]] 
b{1} = [-3.16   -1.4308 -0.2188] 
LW{2,1} = [[ 0.235  -0.9496  0.6662]]  b{2} = [-0.4439]
四點輸出 = [-0.91 -0.95  1.05  0.96] ，mse = 0.0035
NetBpmdl：輸入 [0.62; 0.79] → y = 0.694


In [13]:
g1, g2 = np.linspace(-1.5, 2.5, 61), np.linspace(-0.5, 5.5, 61)
G1, G2 = np.meshgrid(g1, g2)
Z = netbp(np.vstack([G1.ravel(), G2.ravel()])).reshape(G1.shape)
fig = go.Figure(go.Surface(x=g1, y=g2, z=np.round(Z, 3), colorscale="RdBu", reversescale=True, opacity=0.9, colorbar_title="y"))
fig.add_scatter3d(x=Pd[0], y=Pd[1], z=Td, mode="markers", marker=dict(size=6, color="black"), name="training data")
fig.update_layout(title="NetBp.mat output y(p1, p2)", scene=dict(xaxis_title="p1", yaxis_title="p2", zaxis_title="y"), width=620, height=520)
fig.show()

### 自己用 traingd 訓練一次

`traingd` 是批次梯度下降：每個回合把四筆資料的梯度加總後更新一次。下面從 −0.5～0.5 的亂數初值開始（MATLAB 的 newff 預設用 Nguyen–Widrow 初始化 `initnw`，這裡沒有照做），
學習率 0.05、訓練 500 回合。梯度的算法見[倒傳遞網路](https://yoshiagent.github.io/IntControl/NN-NetBp.html#bp)一頁：

In [14]:
def train_gd(P, T, nh=3, lr=0.05, epochs=500, seed=1):
    r = np.random.default_rng(seed)
    W1, b1 = r.uniform(-0.5, 0.5, (nh, 2)), r.uniform(-0.5, 0.5, nh)
    W2, b2 = r.uniform(-0.5, 0.5, (1, nh)), r.uniform(-0.5, 0.5, 1)
    Q, hist = P.shape[1], []
    for ep in range(epochs):
        a1 = tansig(W1 @ P + b1[:, None]); a2 = W2 @ a1 + b2[:, None]
        e = T - a2.ravel()
        hist.append(float(np.mean(e ** 2)))
        s2 = -2 / Q * e[None, :]                        # 輸出層的敏感度
        s1 = (1 - a1 ** 2) * (W2.T @ s2)                # 隱藏層：tansig 的導數 1 − a²
        W2 -= lr * s2 @ a1.T; b2 -= lr * s2.sum(1)
        W1 -= lr * s1 @ P.T;  b1 -= lr * s1.sum(1)
    return (W1, b1, W2, b2), hist

params, hist = train_gd(Pd, Td)
print(f"訓練 500 回合後的 mse = {hist[-1]:.2e}，四點輸出 {np.round(netbp(Pd, *params), 3)}")
fig = go.Figure(go.Scatter(y=hist, mode="lines"))
fig.update_layout(title="traingd: mse vs epoch", xaxis_title="epoch", yaxis_title="mse", yaxis_type="log", height=320)
fig.show()

訓練 500 回合後的 mse = 1.10e-11，四點輸出 [-1. -1.  1.  1.]


不同的初值收斂快慢不同，最後的網路權值也不一樣（同樣能把四點分好），所以不會和 NetBp.mat 的權值相同。網站的
[倒傳遞網路](https://yoshiagent.github.io/IntControl/NN-NetBp.html#try)一頁有可以即時調學習率、換初值的互動訓練；
現代的做法（PyTorch、自動微分）見[深度學習](https://yoshiagent.github.io/IntControl/DL.html)一章。